In [ ]:
# ══════════════════════════════════════════════════════════════════════
# OPTIVER V3.93 — V3.92 refactored mit unified PanelForecaster
#
# Identisch zu V3.92 in jedem Feature-Detail (RMSPE, MLP, Blend, etc.)
# Nur der PanelForecaster wurde aus dem unified panel_forecaster.py importiert
# ══════════════════════════════════════════════════════════════════════

import glob, gc, os, sys, warnings
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import minmax_scale
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold
import lightgbm as lgb

sys.path.insert(0, '/Volumes/Crucial_X9_2TB/Programme/PROGR-254')
from panel_forecaster import PanelConfig, PanelForecaster, MLPRegressor
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')

DATA_DIR   = '/kaggle/input/competitions/optiver-realized-volatility-prediction'
OUTPUT_DIR = '/kaggle/working'
N_NEIGHBORS_LIST = [2, 3, 5, 10, 20, 40]
N_MAX = 40
TARGET_COL = 'target'

# ══════════════════════════════════════════════════════════════════════
# 1. TICK-SIZE PRICE RECOVERY  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def calc_price_from_tick(df):
    arr = sorted(np.unique(df.values.flatten()))
    if len(arr) < 2: return np.nan
    diffs = np.diff(arr)
    threshold = np.quantile(diffs, 0.20)
    small_diffs = diffs[diffs <= threshold]
    tick = np.median(small_diffs) if len(small_diffs) else diffs.min()
    return 0.01 / tick if tick > 0 else np.nan

def calc_prices(book_path, stock_id):
    df = pd.read_parquet(book_path,
        columns=['time_id','ask_price1','ask_price2','bid_price1','bid_price2'])
    prices = (df.groupby('time_id')[['ask_price1','ask_price2','bid_price1','bid_price2']]
              .apply(calc_price_from_tick).reset_index())
    prices.columns = ['time_id','price']; prices['stock_id'] = stock_id
    return prices

def build_price_matrix(data_dir, mode='train'):
    paths = glob.glob(f'{data_dir}/book_{mode}.parquet/*/*.parquet')
    df_files = pd.DataFrame({'book_path': paths})
    df_files['stock_id'] = df_files['book_path'].str.extract(r'stock_id=(\d+)')
    df_files = df_files.dropna(subset=['stock_id'])
    df_files['stock_id'] = df_files['stock_id'].astype(int)
    df_prices = pd.concat(Parallel(n_jobs=4)(
        delayed(calc_prices)(r.book_path, r.stock_id) for _, r in df_files.iterrows()))
    return df_prices.pivot(index='time_id', columns='stock_id', values='price')

# ══════════════════════════════════════════════════════════════════════
# 2. PCA MULTI-METHOD TIME-ID ORDER  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def reconstruct_time_id_order(price_pivot, anchor_stock=61):
    scaled = pd.DataFrame(minmax_scale(price_pivot.fillna(price_pivot.mean())),
                          index=price_pivot.index, columns=price_pivot.columns)
    clf1 = TSNE(n_components=1, perplexity=min(400, len(price_pivot)-1),
                random_state=0, n_iter=2000)
    X1 = clf1.fit_transform(scaled.values)
    clf2 = TSNE(n_components=1, perplexity=50, random_state=0,
                init=X1, n_iter=2000, method='exact')
    X2 = clf2.fit_transform(scaled.values)
    time_ids = price_pivot.index.values
    ordered = time_ids[np.argsort(X2[:, 0])]
    if anchor_stock in price_pivot.columns:
        if price_pivot.loc[ordered[0], anchor_stock] > price_pivot.loc[ordered[-1], anchor_stock]:
            ordered = ordered[::-1]
        anchor_along = price_pivot.loc[ordered, anchor_stock].dropna().values
        mono = np.sum(np.diff(anchor_along) > 0) / (len(anchor_along) - 1)
        print(f'  AMZN monotonicity (2-stage t-SNE): {mono:.3f}')
    return pd.DataFrame({'time_id': ordered})

# ══════════════════════════════════════════════════════════════════════
# 3. BASE FEATURES  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def compute_book_features(book_df):
    features = {}
    df = book_df.copy()
    df['wap1'] = (df['bid_price1']*df['ask_size1'] + df['ask_price1']*df['bid_size1']) / (df['bid_size1']+df['ask_size1'])
    df['wap2'] = (df['bid_price2']*df['ask_size2'] + df['ask_price2']*df['bid_size2']) / (df['bid_size2']+df['ask_size2'])
    df['log_return1'] = np.log(df['wap1'] / df['wap1'].shift(1))
    df['log_return2'] = np.log(df['wap2'] / df['wap2'].shift(1))
    features['book.log_return1.realized_volatility'] = np.sqrt(np.nansum(df['log_return1']**2))
    features['book.log_return2.realized_volatility'] = np.sqrt(np.nansum(df['log_return2']**2))
    features['book.log_return1.mean']     = df['log_return1'].mean()
    features['book.log_return1.std']      = df['log_return1'].std()
    features['book.log_return1.skew']     = df['log_return1'].skew()
    features['book.log_return1.kurtosis'] = df['log_return1'].kurtosis()
    features['book.log_return1.max']      = df['log_return1'].max()
    features['book.log_return1.min']      = df['log_return1'].min()
    features['book.log_return2.mean']     = df['log_return2'].mean()
    features['book.log_return2.std']      = df['log_return2'].std()
    features['book.wap1.mean'] = df['wap1'].mean(); features['book.wap1.std'] = df['wap1'].std()
    features['book.wap2.mean'] = df['wap2'].mean(); features['book.wap2.std'] = df['wap2'].std()
    df['total_volume'] = df['bid_size1']+df['ask_size1']+df['bid_size2']+df['ask_size2']
    features['book.total_volume.mean'] = df['total_volume'].mean()
    features['book.total_volume.std']  = df['total_volume'].std()
    features['book.total_volume.sum']  = df['total_volume'].sum()
    features['book.total_volume.max']  = df['total_volume'].max()
    df['spread1'] = df['ask_price1']-df['bid_price1']
    df['spread2'] = df['ask_price2']-df['bid_price2']
    features['book.spread1.mean'] = df['spread1'].mean()
    features['book.spread1.std']  = df['spread1'].std()
    features['book.spread1.max']  = df['spread1'].max()
    features['book.spread2.mean'] = df['spread2'].mean()
    df['imbalance1'] = (df['bid_size1']-df['ask_size1']) / (df['bid_size1']+df['ask_size1'])
    features['book.imbalance1.mean'] = df['imbalance1'].mean()
    features['book.imbalance1.std']  = df['imbalance1'].std()
    features['book.seconds.count']   = len(df)
    return features

def compute_trade_features(trade_df):
    features = {}
    if trade_df is None or len(trade_df) == 0: return features
    df = trade_df.copy()
    df['log_return'] = np.log(df['price'] / df['price'].shift(1))
    features['trade.log_return.realized_volatility'] = np.sqrt(np.nansum(df['log_return']**2))
    features['trade.log_return.mean'] = df['log_return'].mean()
    features['trade.log_return.std']  = df['log_return'].std()
    features['trade.order_count']     = len(df)
    features['trade.size.sum']  = df['size'].sum()
    features['trade.size.mean'] = df['size'].mean()
    features['trade.size.std']  = df['size'].std()
    features['trade.size.max']  = df['size'].max()
    features['trade.price.mean'] = df['price'].mean()
    features['trade.price.std']  = df['price'].std()
    return features

def create_features_for_stock(stock_id, data_dir, mode='train'):
    book_files = glob.glob(f'{data_dir}/book_{mode}.parquet/stock_id={stock_id}/*.parquet')
    if not book_files: return None
    book_df = pd.read_parquet(book_files[0])
    trade_files = glob.glob(f'{data_dir}/trade_{mode}.parquet/stock_id={stock_id}/*.parquet')
    trade_df = pd.read_parquet(trade_files[0]) if trade_files else None
    results = []
    for tid in book_df['time_id'].unique():
        book_t = book_df[book_df['time_id']  == tid]
        trade_t = trade_df[trade_df['time_id']== tid] if trade_df is not None else None
        f = compute_book_features(book_t); f.update(compute_trade_features(trade_t))
        f['stock_id'] = stock_id; f['time_id'] = tid
        results.append(f)
    return pd.DataFrame(results)

# ══════════════════════════════════════════════════════════════════════
# 4. NN FEATURES  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def build_nn_neighbor_cube(pivot_scaled, feat_pivot, n_max):
    actual_n = min(n_max, len(pivot_scaled))
    nn = NearestNeighbors(n_neighbors=actual_n, p=1); nn.fit(pivot_scaled)
    _, indices = nn.kneighbors(pivot_scaled)
    cube = np.zeros((actual_n, *feat_pivot.shape))
    for i in range(actual_n):
        cube[i,:,:] = feat_pivot.values[indices[:,i], :]
    return cube, actual_n

def make_nn_feature(cube, feat_pivot, n, agg_func, col_name):
    actual_n = cube.shape[0]
    n_use = min(n, actual_n - 1)
    if n_use < 1: return None
    agg_values = agg_func(cube[1:n_use+1,:,:], axis=0)
    agg_df = pd.DataFrame(agg_values, columns=feat_pivot.columns, index=feat_pivot.index)
    dst = agg_df.unstack().reset_index()
    dst.columns = ['stock_id','time_id', f'{col_name}_nn{n}_{agg_func.__name__}']
    return dst

def compute_all_nn_features(df, price_pivot, n_max=40):
    pivot_filled = price_pivot.fillna(price_pivot.mean())
    pivot_scaled = pd.DataFrame(minmax_scale(pivot_filled),
                                index=pivot_filled.index, columns=pivot_filled.columns)
    nn_base_features = [
        'book.log_return1.realized_volatility',
        'book.log_return2.realized_volatility',
        'trade.log_return.realized_volatility',
        'book.total_volume.sum','book.total_volume.mean',
        'trade.size.sum','trade.order_count',
        'book.spread1.mean','book.log_return1.std','book.log_return1.mean',
        'book.imbalance1.mean','trade.size.mean',
    ]
    nn_base_features = [f for f in nn_base_features if f in df.columns]
    all_nn_dfs = []
    for f_col in nn_base_features:
        feat_pivot = df.pivot(index='time_id', columns='stock_id', values=f_col)
        feat_pivot = feat_pivot.reindex(pivot_scaled.index).fillna(feat_pivot.mean())
        common = pivot_scaled.columns.intersection(feat_pivot.columns)
        feat_pivot_a = feat_pivot[common]; pivot_scaled_a = pivot_scaled[common]
        cube, _ = build_nn_neighbor_cube(pivot_scaled_a, feat_pivot_a, n_max)
        for n in N_NEIGHBORS_LIST:
            for agg in [np.mean, np.std]:
                d = make_nn_feature(cube, feat_pivot_a, n, agg, f_col)
                if d is not None: all_nn_dfs.append(d)
        del cube; gc.collect()
    result = df[['stock_id','time_id']].copy()
    for d in all_nn_dfs:
        result = result.merge(d, on=['stock_id','time_id'], how='left')
    return result

def compute_stock_nn_features(df, n_max=40):
    nn_bases = ['book.log_return1.realized_volatility',
                'trade.log_return.realized_volatility',
                'book.total_volume.mean']
    nn_bases = [f for f in nn_bases if f in df.columns]
    rv_pivot = df.pivot(index='time_id', columns='stock_id',
                        values='book.log_return1.realized_volatility')
    rv_pivot = rv_pivot.fillna(rv_pivot.mean())
    stock_profile = rv_pivot.T
    stock_profile_scaled = pd.DataFrame(minmax_scale(stock_profile.fillna(0)),
                                        index=stock_profile.index,
                                        columns=stock_profile.columns)
    actual_n = min(n_max, len(stock_profile_scaled))
    nn = NearestNeighbors(n_neighbors=actual_n, p=1); nn.fit(stock_profile_scaled)
    _, indices = nn.kneighbors(stock_profile_scaled)
    all_nn_dfs = []
    for f_col in nn_bases:
        feat_pivot = df.pivot(index='time_id', columns='stock_id', values=f_col)
        feat_pivot = feat_pivot.fillna(feat_pivot.mean())
        feat_by_stock = feat_pivot.T
        common = stock_profile_scaled.index.intersection(feat_by_stock.index)
        feat_aligned = feat_by_stock.reindex(common)
        cube = np.zeros((actual_n, len(common), feat_aligned.shape[1]))
        for i in range(actual_n):
            cube[i,:,:] = feat_aligned.values[indices[:len(common), i], :]
        for n in [3,5,10,20]:
            n_use = min(n, actual_n-1)
            if n_use < 1: continue
            agg_mean = np.nanmean(cube[1:n_use+1,:,:], axis=0)
            agg_df = pd.DataFrame(agg_mean, index=common, columns=feat_aligned.columns).T
            agg_df = agg_df.unstack().reset_index()
            agg_df.columns = ['stock_id','time_id', f'{f_col}_stock_nn{n}_mean']
            all_nn_dfs.append(agg_df)
        del cube; gc.collect()
    result = df[['stock_id','time_id']].copy()
    for d in all_nn_dfs:
        result = result.merge(d, on=['stock_id','time_id'], how='left')
    return result

# ══════════════════════════════════════════════════════════════════════
# 5. τ-SHIFT FEATURES  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def add_tau_shift_features(combined_df, time_id_order, base_features,
                           lags=(-3,-2,-1,1,2,3,5)):
    ordered_tids = time_id_order['time_id'].values
    for f_col in base_features:
        if f_col not in combined_df.columns:
            print(f'    WARN: {f_col} not in df, skipping τ-shift')
            continue
        pivot = combined_df.pivot(index='time_id', columns='stock_id', values=f_col)
        pivot_ordered = pivot.reindex(ordered_tids)
        for lag in lags:
            shifted = pivot_ordered.shift(-lag)
            stacked = shifted.unstack().reset_index()
            sign = '+' if lag > 0 else ''
            stacked.columns = ['stock_id','time_id', f'{f_col}_tau{sign}{lag}']
            combined_df = combined_df.merge(stacked, on=['stock_id','time_id'], how='left')
    return combined_df

# ══════════════════════════════════════════════════════════════════════
# 6. OOF STOCK-TARGET-ENCODING  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def add_stock_target_features(train_df, test_df, target_col='target',
                              n_splits=5, seed=42):
    stat_funcs = {
        'mean':   lambda s: s.mean(),
        'std':    lambda s: s.std(),
        'median': lambda s: s.median(),
        'q05':    lambda s: s.quantile(0.05),
        'q25':    lambda s: s.quantile(0.25),
        'q75':    lambda s: s.quantile(0.75),
        'q95':    lambda s: s.quantile(0.95),
    }
    feat_names = [f'stock_target_{n}' for n in stat_funcs]
    for fn in feat_names: train_df[fn] = np.nan
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in kf.split(train_df):
        tr_part = train_df.iloc[tr_idx]
        for name, func in stat_funcs.items():
            stats    = tr_part.groupby('stock_id')[target_col].apply(func)
            global_v = func(tr_part[target_col])
            mapped   = train_df.iloc[va_idx]['stock_id'].map(stats).fillna(global_v)
            train_df.loc[train_df.index[va_idx], f'stock_target_{name}'] = mapped.values
    if len(test_df) > 0:
        for name, func in stat_funcs.items():
            stats    = train_df.groupby('stock_id')[target_col].apply(func)
            global_v = func(train_df[target_col])
            test_df[f'stock_target_{name}'] = test_df['stock_id'].map(stats).fillna(global_v)
    return train_df, test_df

# ══════════════════════════════════════════════════════════════════════
# 7. PROCESS FEATURES + CV + RMSPE  (identisch zu V3.8)
# ══════════════════════════════════════════════════════════════════════
def process_features(df):
    rank_cols = [c for c in df.columns if 'trade.order_count' in c or 'book.total_volume' in c]
    for col in rank_cols:
        if col in df.columns and df[col].notna().any():
            df[f'{col}_rank'] = df.groupby('time_id')[col].rank(pct=True)
    log_cols = [c for c in df.columns if any(k in c for k in ['volume.sum','size.sum','order_count'])]
    for col in log_cols:
        if col in df.columns and not col.endswith('_rank') and not col.endswith('_log'):
            df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))
    return df

def time_series_cv(df, time_id_order, n_folds=4, val_ratio=0.1):
    ordered = time_id_order['time_id'].values
    valid = [t for t in ordered if t in df['time_id'].values]
    n = len(valid); splits = []
    for fold in range(n_folds):
        val_size = int(n * val_ratio)
        val_start = n - val_size * (fold+1)
        if val_start < 0: continue
        val_tids = valid[val_start:val_start+val_size]
        train_tids = valid[:val_start]
        tr_idx = df[df['time_id'].isin(train_tids)].index
        va_idx = df[df['time_id'].isin(val_tids)].index
        if len(tr_idx) > 0 and len(va_idx) > 0:
            splits.append((tr_idx, va_idx))
    return splits

def rmspe(y_true, y_pred):
    return float(np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2)))

# ══════════════════════════════════════════════════════════════════════
# 10. MAIN PIPELINE — V3.93 mit unified PanelForecaster
# ══════════════════════════════════════════════════════════════════════
print('='*70)
print('V3.93 = V3.92 refactored mit unified PanelForecaster')
print('='*70)

# ── [1/9] Price Matrices ──
print('\\n[1/9] Building price matrices (train + test) and combining...')
price_pivot_train = build_price_matrix(DATA_DIR, 'train')
print(f'  train price pivot: {price_pivot_train.shape}')

test_parquet_exists = len(glob.glob(f'{DATA_DIR}/book_test.parquet/*/*.parquet')) > 0
if test_parquet_exists:
    price_pivot_test = build_price_matrix(DATA_DIR, 'test')
    print(f'  test  price pivot: {price_pivot_test.shape}')
    combined_price = pd.concat([price_pivot_train, price_pivot_test])
    combined_price = combined_price[~combined_price.index.duplicated(keep='first')]
    combined_price = combined_price.fillna(combined_price.mean())
else:
    combined_price = price_pivot_train.copy()
print(f'  combined price pivot: {combined_price.shape}')

# ── [2/9] Time-ID Order ──
print('\\n[2/9] Reconstructing time-id order (2-stage t-SNE)...')
time_id_order = reconstruct_time_id_order(combined_price, anchor_stock=61)
print(f'  ordered time_ids: {len(time_id_order)}')

# ── [3/9] Base Features ──
print('\\n[3/9] Computing base features for train + test...')
train_csv = pd.read_csv(f'{DATA_DIR}/train.csv')
test_csv  = pd.read_csv(f'{DATA_DIR}/test.csv')
train_stock_ids = sorted(train_csv['stock_id'].unique())
print(f'  Train stocks: {len(train_stock_ids)}, time_ids: {train_csv["time_id"].nunique()}')

train_feats = Parallel(n_jobs=-1)(
    delayed(create_features_for_stock)(s, DATA_DIR, 'train') for s in train_stock_ids)
train_feats = [f for f in train_feats if f is not None]
train_features_df = pd.concat(train_feats, ignore_index=True)
train_df = train_csv.merge(train_features_df, on=['stock_id','time_id'], how='left')
print(f'  train_df: {train_df.shape}')

if len(test_csv) > 0 and test_parquet_exists:
    test_stock_ids = sorted(test_csv['stock_id'].unique())
    test_feats = Parallel(n_jobs=-1)(
        delayed(create_features_for_stock)(s, DATA_DIR, 'test') for s in test_stock_ids)
    test_feats = [f for f in test_feats if f is not None]
    if test_feats:
        test_features_df = pd.concat(test_feats, ignore_index=True)
        test_df = test_csv.merge(test_features_df, on=['stock_id','time_id'], how='left')
    else:
        test_df = test_csv.copy()
    print(f'  test_df:  {test_df.shape}')
else:
    test_df = test_csv.copy()
    print('  test_df: empty')

# ── [4/9] NN-Features + τ-Shift auf COMBINED ──
print('\\n[4/9] Computing NN + τ-shift on COMBINED data...')
train_df['_src'] = 'train'
if len(test_df) > 0:
    test_df['_src'] = 'test'
    if TARGET_COL not in test_df.columns:
        test_df[TARGET_COL] = np.nan
    combined_df = pd.concat([train_df, test_df], ignore_index=True)
else:
    combined_df = train_df.copy()
print(f'  combined_df shape: {combined_df.shape}')

print('  computing time-NN...')
nn_time_combined = compute_all_nn_features(combined_df, combined_price, n_max=N_MAX)
combined_df = combined_df.merge(nn_time_combined,
                                on=['stock_id','time_id'], how='left',
                                suffixes=('','_nn'))
print(f'  after time-NN:  {combined_df.shape}')

print('  computing stock-NN...')
nn_stock_combined = compute_stock_nn_features(combined_df, n_max=N_MAX)
combined_df = combined_df.merge(nn_stock_combined,
                                on=['stock_id','time_id'], how='left',
                                suffixes=('','_snn'))
print(f'  after stock-NN: {combined_df.shape}')

print('  adding τ-shift features...')
tau_base_features = [
    'book.log_return1.realized_volatility',
    'book.log_return2.realized_volatility',
    'trade.log_return.realized_volatility',
    'book.total_volume.mean',
    'book.spread1.mean',
    'book.log_return1.std',
]
combined_df = add_tau_shift_features(
    combined_df, time_id_order, tau_base_features,
    lags=(-3, -2, -1, 1, 2, 3, 5))
print(f'  after τ-shift:  {combined_df.shape}')

combined_df = process_features(combined_df)

df = combined_df[combined_df['_src'] == 'train'].drop(columns=['_src']).reset_index(drop=True)
if (combined_df['_src'] == 'test').any():
    test_df = combined_df[combined_df['_src'] == 'test'].drop(columns=['_src']).reset_index(drop=True)
else:
    test_df = test_df.drop(columns=['_src'], errors='ignore')
print(f'  → train: {df.shape}, test: {test_df.shape}')

# ── [5/9] OOF Stock-Target-Encoding ──
print('\\n[5/9] OOF stock-target features...')
df, test_df = add_stock_target_features(df, test_df, target_col=TARGET_COL,
                                        n_splits=5, seed=42)
print('  added 7 stock_target_* features')

# ── [6/9] Feature-Matrix ──
print('\\n[6/9] Building feature matrix...')
exclude = ['stock_id','time_id',TARGET_COL,'row_id','_src']
feature_cols = [c for c in df.columns if c not in exclude]
print(f'  features: {len(feature_cols)}')

X = np.nan_to_num(df[feature_cols].fillna(0).values.astype(np.float32),
                  nan=0, posinf=0, neginf=0)
y = df[TARGET_COL].values.astype(np.float32)
splits = time_series_cv(df, time_id_order, n_folds=4, val_ratio=0.1)
print(f'  folds: {len(splits)}')














# ── [7/9] Training mit PanelForecaster + MLPForecaster ──
print('\\n[7/9] Training PanelForecaster (LGB, RMSPE-weights) + MLPForecaster...')
# ── [7/9] Training with unified PanelForecaster + MLPRegressor ──
print("\n[7/9] Training with unified PanelForecaster (LGB, RMSPE) + MLPRegressor...")

lgb_params = dict(
    verbose=-1, n_estimators=3000, learning_rate=0.05,
    num_leaves=127, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    lambda_l1=0.1, lambda_l2=1.0,
)
panel_models = [
    ("lgbm", LGBMRegressor, lgb_params),
    ("mlp",  MLPRegressor,  dict(n_epochs=30, dropout1=0.3, dropout2=0.2, verbose=False)),
]
panel_cfg = PanelConfig(date_col="time_id", target_col=TARGET_COL, entity_cols=["stock_id"])
panel = PanelForecaster(panel_cfg, models=panel_models, log_target=False,
                        sample_weight_mode="rmspe", precomputed_features=True,
                        n_seeds=1, verbose=True, ensemble_method="blend",
                        n_folds=4, val_ratio=0.1, early_stopping_rounds=150)
panel.fit((X, y))

# ── [9/9] Test-Predictions ──
print('\\n[9/9] Test predictions...')
if len(test_df) > 0 and TARGET_COL in test_df.columns:
    for c in feature_cols:
        if c not in test_df.columns: test_df[c] = 0.0
    X_test = np.nan_to_num(test_df[feature_cols].fillna(0).values.astype(np.float32),
                           nan=0, posinf=0, neginf=0)
    final = panel.predict(X_test)

    sub = test_df[['stock_id','time_id']].copy()
    sub['target'] = final
    sub['row_id'] = sub['stock_id'].astype(str) + '-' + sub['time_id'].astype(str)
    sub = sub[['row_id','target']]
else:
    sub = pd.DataFrame({'row_id': ['0-4','0-32','0-34'],
                        'target': [0.004, 0.002, 0.003]})

sub.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)
print(f'\nSubmission: {sub.shape}')
print(sub.head(10))
print('Fertig — V3.93 (unified PanelForecaster, 0.20514-Pipeline)!')